# TruthChain — fine-tune the GPU models on Colab

Trains **Model 3** (DistilBERT, fake news) and **Model 4** (EfficientNet-B0, deepfake faces).

1. **Runtime → Change runtime type → T4 GPU**
2. On your PC run `python training/make_colab_bundle.py --with-faces`, which creates `training/colab_bundle.zip`
   and `training/colab_faces.zip` (your split face dataset, so models 2 and 4 are compared on the same data)
3. Get `kaggle.json` from kaggle.com → Settings → API → *Create New Token*
4. Run the cells top to bottom (about 1 hour in total). The last cell downloads `trained_models.zip`.

In [ ]:
!nvidia-smi -L   # must show a GPU

## 1. Upload `colab_bundle.zip`, `kaggle.json` and `colab_faces.zip`

In [ ]:
from google.colab import files
uploaded = files.upload()   # select colab_bundle.zip, kaggle.json and (optionally) colab_faces.zip

In [ ]:
!unzip -o -q colab_bundle.zip
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!ls training

## 2. Download the datasets from Kaggle

In [ ]:
import os
!kaggle datasets download -d saurabhshahane/fake-news-classification -p data --unzip
if os.path.exists("colab_faces.zip"):        # your own split dataset (same as model 2)
    !unzip -o -q colab_faces.zip
    FACES = "data/faces"
else:                                         # fallback: the 140k Real and Fake Faces dataset
    !kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p data --unzip
    FACES = "data/real_vs_fake/real-vs-fake"
!ls data {FACES}

## 3. Model 3: fine-tune DistilBERT on WELFake (~25 min)

In [ ]:
!python training/text_finetune.py --data data/WELFake_Dataset.csv

## 4. Model 4: fine-tune EfficientNet-B0 on the face dataset (~15–30 min)

In [ ]:
!python training/image_finetune.py --data {FACES} --workers 2

## 5. Results and download

In [ ]:
!python training/compare.py
from IPython.display import Image, display
for m in ["text-distilbert", "image-efficientnet"]:
    display(Image(f"models/{m}/confusion_matrix.png"), Image(f"models/{m}/training_curve.png"))

In [ ]:
!zip -r -q trained_models.zip models
files.download("trained_models.zip")

Unzip `trained_models.zip` into the project root, so you get `project1/models/text-distilbert` and `project1/models/image-efficientnet`. Then restart the gateway, which picks them up automatically.